In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet.csv"

# Preprocessing

In [3]:
from sklearn.ensemble import GradientBoostingRegressor

def impute_feed_predictively(df_train, df_test):
    """
    Trains a model to predict missing Feed_Quantity_kg based on other features.
    Returns updated train and test dataframes with imputed feed values.
    """
    # Features to use for predicting feed
    feed_predictors = [
        "Weight_kg", "Age_Months", "Days_in_Milk", 
        "Water_Intake_L", "Previous_Week_Avg_Yield",
        "Ambient_Temperature_C", "Lactation_Stage", 
        "Parity", "Date"  # Date will be season after preprocessing
    ]
    
    # Identify missing feed in train
    train_missing_mask = df_train["Feed_Quantity_kg"].isna()
    
    if train_missing_mask.sum() == 0:
        print("No missing feed data in training set")
        return df_train, df_test
    
    print(f"\nPredictive Feed Imputation:")
    print(f"Train samples with missing feed: {train_missing_mask.sum()} ({100*train_missing_mask.mean():.1f}%)")
    
    # Samples WITH feed data
    has_feed = df_train[~train_missing_mask].copy()
    missing_feed = df_train[train_missing_mask].copy()
    
    # Prepare features for imputation (one-hot encode categoricals)
    X_feed_train = pd.get_dummies(has_feed[feed_predictors], drop_first=True)
    y_feed_train = has_feed["Feed_Quantity_kg"].values
    
    # Train feed imputation model
    feed_imputer = GradientBoostingRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42
    )
    feed_imputer.fit(X_feed_train, y_feed_train)
    
    # Predict missing feed in TRAINING set
    X_feed_predict = pd.get_dummies(missing_feed[feed_predictors], drop_first=True)
    X_feed_predict = X_feed_predict.reindex(columns=X_feed_train.columns, fill_value=0)
    predicted_feed_train = feed_imputer.predict(X_feed_predict)
    
    print(f"Predicted train feed range: {predicted_feed_train.min():.2f} - {predicted_feed_train.max():.2f} kg")
    
    # Update training data
    df_train = df_train.copy()
    df_train.loc[train_missing_mask, "Feed_Quantity_kg"] = predicted_feed_train
    
    # Predict missing feed in TEST set
    test_missing_mask = df_test["Feed_Quantity_kg"].isna()
    
    if test_missing_mask.sum() > 0:
        print(f"Test samples with missing feed: {test_missing_mask.sum()} ({100*test_missing_mask.mean():.1f}%)")
        
        missing_feed_test = df_test[test_missing_mask].copy()
        X_feed_predict_test = pd.get_dummies(missing_feed_test[feed_predictors], drop_first=True)
        X_feed_predict_test = X_feed_predict_test.reindex(columns=X_feed_train.columns, fill_value=0)
        predicted_feed_test = feed_imputer.predict(X_feed_predict_test)
        
        print(f"Predicted test feed range: {predicted_feed_test.min():.2f} - {predicted_feed_test.max():.2f} kg")
        
        df_test = df_test.copy()
        df_test.loc[test_missing_mask, "Feed_Quantity_kg"] = predicted_feed_test
    
    return df_train, df_test

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler, MinMaxScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score",
    "Breed",
    "Body_Condition_Score",


]

CATEGORICAL_FEATURES = [
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs",
    "Young",
    # "IBR_Vaccine",
    # "Anthrax_Vaccine",
    # "Rabies_Vaccine"
]

STANDARD_SCALED_FEATURES = [
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield",
]

def preprocess (
    dtrain, dtest
):
    """
    NOTES:
    - Interaction features do not help
    - Squaring feed does not help
    - clipping negative records worsens results
    - RobustScaler performs similarly to standard scaler
    """
    # Convert month to season
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Breed
    dtrain['Young'] = (dtrain['Age_Months'] < 60).astype(int)
    dtest['Young'] = (dtest['Age_Months'] < 60).astype(int)

    # Imputation
    # dtrain, dtest = impute_feed_predictively(dtrain, dtest)    
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    
    # Drop features deemed unnecessary
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # One-hot encode
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # Standardize data
    scaler = StandardScaler ()
    # TODO: For now, standardize everything. In future, see if min/max scaling is better for non-gaussian data
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (
                                                dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (
                                                dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

# NOTE: Dropping negative records worsens results
train_data = pd.read_csv (TRAIN_PATH)

X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

X_train, X_test, scaler = preprocess (X_train, X_test)

print (X_train.head ())

nan_cols = X_train.columns[X_train.isna ().any ()]

        Age_Months  Weight_kg    Parity  Days_in_Milk  Feed_Quantity_kg  \
84926     0.907972   0.705222  0.291635      0.132346         -0.993314   
23359     0.301491  -0.670405 -0.294678     -1.467179          0.344349   
92153     0.994613  -1.126181 -0.294678     -1.124423          0.319565   
126952    1.658854  -0.328746  1.464261     -0.039031         -1.561874   
55765    -1.286914  -0.525166  0.877948     -1.400532         -0.003972   

        Water_Intake_L  Ambient_Temperature_C  Anthrax_Vaccine  IBR_Vaccine  \
84926         0.359810              -0.463694                0            1   
23359         0.386087               0.729661                1            0   
92153        -0.775817               0.727743                0            1   
126952       -2.629058              -0.176845                1            0   
55765         0.002619               0.319088                0            0   

        Rabies_Vaccine  Previous_Week_Avg_Yield  Mastitis  \
84926        

# Training Set Eval

In [5]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.base import clone

# RMSE at each checkpoint
train_rmse_list = []
test_rmse_list = []

total_iterations = 0
iterations_per_step = 10
model_template = MLPRegressor (hidden_layer_sizes = (100, 100, 100),
                               activation = "tanh",
                               learning_rate_init = 0.00003,
                               learning_rate = "adaptive",
                               # alpha = 0.01,
                               early_stopping = False, 
                               # validation_fraction = 0.15,
                               n_iter_no_change = 20,
                               verbose = False,
                               warm_start = True,
                               max_iter = iterations_per_step,
                               random_state = 1)

In [6]:
model = clone (model_template)
prev_rmse = float ('inf')
test_rmse = 0
TOLERANCE = 0.00005
print (model)
# total_iterations < 200
while (test_rmse + TOLERANCE < prev_rmse):
    if test_rmse > 0:
        prev_rmse = test_rmse
    model.fit (X_train, y_train)

    # Compute RMSE
    y_train_pred = model.predict (X_train)
    y_test_pred = model.predict (X_test)
    train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
    test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

    train_rmse_list.append (train_rmse)
    test_rmse_list.append (test_rmse)

    total_iterations += iterations_per_step
    print (f"Iteration {total_iterations}: Train RMSE={train_rmse:.5f}, Test RMSE={test_rmse:.5f}")

print ("Termination Condition Reached:")
print (f"Prev: {prev_rmse}")
print (f"Last: {test_rmse}", flush = True)


MLPRegressor(activation='tanh', hidden_layer_sizes=(100, 100, 100),
             learning_rate='adaptive', learning_rate_init=3e-05, max_iter=10,
             n_iter_no_change=20, random_state=1, warm_start=True)
Iteration 10: Train RMSE=4.16574, Test RMSE=4.16572
Iteration 20: Train RMSE=4.12603, Test RMSE=4.12317
Iteration 30: Train RMSE=4.11647, Test RMSE=4.11327
Iteration 40: Train RMSE=4.11203, Test RMSE=4.10909
Iteration 50: Train RMSE=4.10941, Test RMSE=4.10696
Iteration 60: Train RMSE=4.10759, Test RMSE=4.10569
Iteration 70: Train RMSE=4.10620, Test RMSE=4.10489
Iteration 80: Train RMSE=4.10508, Test RMSE=4.10437
Iteration 90: Train RMSE=4.10414, Test RMSE=4.10403
Iteration 100: Train RMSE=4.10333, Test RMSE=4.10382
Iteration 110: Train RMSE=4.10260, Test RMSE=4.10371
Iteration 120: Train RMSE=4.10194, Test RMSE=4.10366
Iteration 130: Train RMSE=4.10132, Test RMSE=4.10366
Termination Condition Reached:
Prev: 4.1036582152639784
Last: 4.103657295290459


In [7]:
# Predict on all data for feedback
raw_data = pd.read_csv (TRAIN_PATH)

y_pred = model.predict (X_train)
rmse = np.sqrt (mean_squared_error (y_train, y_pred))
print ("Train RMSE:", rmse)
print (f"Train True Mean: {y_train.mean ()}")
print (f"Train Pred Mean: {y_pred.mean ()}")
# diff = y_train.mean () - y_pred.mean ()
# print (f"Diff: {diff}")

# print (f"STD true: {y_train.std ()}")
# print (f"STD diff: {(y_train - y_pred).std ()}")

y_pred = model.predict (X_test)
rmse = np.sqrt (mean_squared_error (y_test, y_pred))
print ("Test RMSE:", rmse)

# y_pred = y_pred + diff
# nudge_rmse = np.sqrt (mean_squared_error (y_test, y_pred))
# print ("Nudged Test RMSE:", nudge_rmse)
# print (f"Gain: {rmse - nudge_rmse}")

# Season check (summer should very good)
for season in ["Spring", "Summer", "Winter"]:
    mask = X_train[f"Date_{season}"]
    y_true_cls = y_train[mask]
    y_pred_cls = model.predict(X_train[mask])
    rmse_cls = np.sqrt(mean_squared_error(y_true_cls, y_pred_cls))
    print(f"{season} RMSE: {rmse_cls:.4f}")

# Fall: where all other season columns are 0
fall_mask = ~(X_train["Date_Spring"] | X_train["Date_Summer"] | X_train["Date_Winter"])
y_true_fall = y_train[fall_mask]
y_pred_fall = model.predict(X_train[fall_mask])
rmse_fall = np.sqrt(mean_squared_error(y_true_fall, y_pred_fall))
print(f"Fall RMSE: {rmse_fall:.4f}")


Train RMSE: 4.1013188414405075
Train True Mean: 15.584066685700828
Train Pred Mean: 15.60677334869247
Test RMSE: 4.103657295290459
Spring RMSE: 4.2591
Summer RMSE: 3.8179
Winter RMSE: 4.2017
Fall RMSE: 4.1129


In [8]:
# # See what was good and bad
# errors = np.sqrt ((y_test - y_pred) ** 2)
# df_results = X_test.copy ()
# scaled_part = df_results[STANDARD_SCALED_FEATURES]
# scaled_inverse = pd.DataFrame (scaler.inverse_transform (scaled_part),
#                                columns = STANDARD_SCALED_FEATURES,
#                                index = df_results.index,)

# # Replace only those columns
# df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
# df_results["y_true"] = y_test
# df_results["y_pred"] = y_pred
# df_results["rmse"] = errors

# df_results = df_results.merge(
#     raw_data,     # <-- this contains ALL original columns
#     left_index=True,
#     right_index=True,
#     how="left"
# )

# print ("\nTop 5 BEST predictions:")
# print(df_results.nsmallest (5, "rmse"))

# print ("\nTop 5 WORST predictions:")
# print (df_results.nlargest (5, "rmse"))

# Final Model

In [9]:
# Build final model
train_data = pd.read_csv (TRAIN_PATH)
test_data = pd.read_csv (TEST_PATH)

X_train = train_data.drop (TARGET_FEATURE, axis = 1)
y_train = train_data[TARGET_FEATURE]

X_test = test_data
X_train, X_test, scaler = preprocess (X_train, X_test)

model = clone (model_template)
model.max_iter = total_iterations
model.fit (X_train, y_train)

,loss,'squared_error'
,hidden_layer_sizes,"(100, ...)"
,activation,'tanh'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'adaptive'
,learning_rate_init,3e-05
,power_t,0.5
,max_iter,130
,shuffle,True


In [10]:
# Final Predictions
y_pred = model.predict (X_test)

print (y_pred.mean ())
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (y_pred) + 1),
                          'Milk_Yield_L': y_pred})
out_data.to_csv (OUT_PATH, index = False)

15.57466841881084
